# 基于LBPH的人脸识别

1. 代码整体结构
实现一个简单的人脸识别系统，主要流程包括：

- 1.加载训练图像与对应标签
- 2.创建识别器
- 3.训练模型
- 4.预测新图像
- 5.输出结果

In [ ]:
import cv2                  # OpenCV计算机视觉库
import numpy as np            # NumPy数值计算库
import os                     # 操作系统接口，用于文件路径和目录遍历

## 1.加载训练图像与标签

In [ ]:
# 数据集路径（需包含子文件夹，每个子文件夹为一个类别）
dataset_path = 'path/to/faces'
faces = []   # 存储所有人脸图像
labels = []  # 存储对应的类别标签

# enumerate自动为每个子文件夹分配从0开始的整数标签
for label, person in enumerate(os.listdir(dataset_path)):
    person_path = os.path.join(dataset_path, person)  # 拼接完整路径
    for img_name in os.listdir(person_path):
        img_path = os.path.join(person_path, img_name)
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)  # 以灰度模式读取
        img = cv2.resize(img, (100, 100))  # 统一尺寸为100x100，LBPH要求所有图像尺寸一致
        faces.append(img)
        labels.append(label)
faces = np.array(faces, dtype=np.uint8)   # 转为NumPy数组
labels = np.array(labels)

In [ ]:
import cv2
import numpy as np
import os

# 使用ORL人脸数据集的正确路径（包含40个人各10张照片）
dataset_path = "img/ORLdataset"

faces = []
labels = []

valid_ext = [".bmp", ".jpg", ".jpeg", ".png", ".pgm"]  # 允许的图片扩展名

for label, person in enumerate(os.listdir(dataset_path)):
    person_path = os.path.join(dataset_path, person)

    if not os.path.isdir(person_path):  # 跳过非文件夹（如.DS_Store）
        continue

    for img_name in os.listdir(person_path):

        ext = os.path.splitext(img_name)[1].lower()  # 获取文件扩展名
        if ext not in valid_ext:
            print("跳过非图片文件:", img_name)
            continue

        img_path = os.path.join(person_path, img_name)
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)  # 灰度模式读取

        if img is None:  # 防止读取失败（如macOS的._开头的元数据文件）
            print("读取失败:", img_path)
            continue

        img = cv2.resize(img, (100, 100))  # 统一尺寸为100x100
        faces.append(img)
        labels.append(label)  # 每个子文件夹对应一个整数标签

faces = np.array(faces, dtype=np.uint8)
labels = np.array(labels)

print("读取成功的图像数量:", len(faces))
print("标签数量:", len(labels))

## 2. 创建识别器

In [ ]:
# 创建LBPH（Local Binary Pattern Histograms）人脸识别器
# threshold=80：置信度上限阈值，distance大于80视为"无法识别"
recognizer = cv2.face.LBPHFaceRecognizer_create(threshold=80)

print("识别器创建成功：", recognizer)

## 3.训练模型

In [ ]:
# 使用faces（图像数组）和labels（标签数组）训练识别器
recognizer.train(faces, labels)
print("模型训练完成！")

## 4.预测新图像

In [ ]:
# 从ORL数据集中选取一张测试图片进行预测
test_img_path = os.path.join("img/ORLdataset", "s1", "1.bmp")  # 第1个人的第1张图

# 读入并预处理：灰度模式 + 统一尺寸为100x100
test_img = cv2.imread(test_img_path, cv2.IMREAD_GRAYSCALE)
test_img = cv2.resize(test_img, (100, 100))

# predict返回：pred_label=预测的类别编号，confidence=与最匹配类别的距离
pred_label, confidence = recognizer.predict(test_img)

print("预测得到的类别编号：", pred_label)
print("对应的距离（置信度，越小越像）：", confidence)

## 5.输出结果

In [ ]:
# 将数字标签转换为可读的人名（ORL数据集中s1~s40表示40个人）
pred_person = f"s{pred_label + 1}"  # 标签从0开始，文件夹名从s1开始

print("预测结果：", pred_person)
print("置信度（越小越像）：", confidence)

In [ ]:
import cv2
import numpy as np
import os

dataset_path = "img/ORLdataset"

faces = []
labels = []
test_faces = []    # 存储测试集图像
test_labels = []   # 存储测试集标签

valid_ext = [".bmp"]   # 只用bmp格式，跳过macOS的._开头的元数据文件

for label, person in enumerate(os.listdir(dataset_path)):
    person_path = os.path.join(dataset_path, person)

    if not os.path.isdir(person_path):
        continue

    # 过滤出干净的图片列表：不以._开头，后缀是.bmp
    img_files = [
        f for f in os.listdir(person_path)
        if (not f.startswith("._")) and os.path.splitext(f)[1].lower() in valid_ext
    ]
    img_files = sorted(img_files)   # 排序保证1.bmp~10.bmp顺序一致

    if len(img_files) == 0:
        print("这个人没有有效图片：", person_path)
        continue

    # 每个人的最后一张图作为测试集，其余作为训练集
    test_img_name = img_files[-1]
    test_img_path = os.path.join(person_path, test_img_name)
    test_img = cv2.imread(test_img_path, 0)  # 0表示灰度模式
    test_img = cv2.resize(test_img, (100, 100))
    test_faces.append(test_img)
    test_labels.append(label)

    # 其余图片作为训练样本
    for img_name in img_files[:-1]:
        img_path = os.path.join(person_path, img_name)
        img = cv2.imread(img_path, 0)
        if img is None:
            print("训练图读取失败：", img_path)
            continue
        img = cv2.resize(img, (100, 100))
        faces.append(img)
        labels.append(label)

faces = np.array(faces, dtype=np.uint8)
labels = np.array(labels)
test_faces = np.array(test_faces, dtype=np.uint8)
test_labels = np.array(test_labels)

print("训练集数量：", len(faces))

In [ ]:
# 重新创建LBPH识别器（不设threshold阈值，让所有预测都返回结果）
recognizer = cv2.face.LBPHFaceRecognizer_create()

recognizer.train(faces, labels)  # 用训练集重新训练
print("模型训练完成！")

In [ ]:
# 预测测试集中的第1张图像（idx=0表示第一个人的最后一张照片）
idx = 0
test_img = test_faces[idx]
true_label = test_labels[idx]  # 真实标签

pred_label, confidence = recognizer.predict(test_img)  # LBPH预测

print("真实类别：", f"s{true_label + 1}")
print("预测类别：", f"s{pred_label + 1}")
print("置信度：", confidence)  # 置信度越大表示越不匹配

In [ ]:
from matplotlib import pyplot as plt
import cv2

# 将灰度图转为BGR以便在上面绘制文字
display = cv2.cvtColor(test_img, cv2.COLOR_GRAY2BGR)
# 在图像底部添加预测结果文字
cv2.putText(display, f"Pred: s{pred_label+1}", (5, 95),
            cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,255,255), 2)

# BGR转RGB后用matplotlib显示
plt.imshow(cv2.cvtColor(display, cv2.COLOR_BGR2RGB))
plt.title(f"True: s{true_label+1}  Pred: s{pred_label+1}\nConf: {confidence:.2f}")  # 标题显示真实/预测类别和置信度
plt.axis('off')
plt.show()

In [ ]:
# 计算整体识别准确率：遍历所有测试图像进行预测并统计正确数
correct = 0

for i in range(len(test_faces)):
    pred_label, confidence = recognizer.predict(test_faces[i])
    if pred_label == test_labels[i]:  # 预测标签与真实标签一致则计数+1
        correct += 1

accuracy = correct / len(test_faces) * 100  # 计算准确率百分比

print(f"测试集数量：{len(test_faces)}")
print(f"预测正确：{correct}")
print(f"识别率 Accuracy：{accuracy:.2f}%")